# 03 — Feature Engineering

**Goal:** Compute the two *derived* features (moisture trend, hour-of-day) and assemble the
full six-feature table for every valid cycle:

1. Temperature (direct)
2. Humidity (direct)
3. Soil moisture, instantaneous (direct)
4. **Moisture trend** — Δ over the last 2–3 hourly readings (derived, this notebook)
5. Light intensity (direct — BH1750)
6. **Hour of day** — 0–23, extracted from timestamp (derived, this notebook)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PREPROCESSED_DIR = Path("../data/preprocessed")

manifest = pd.read_csv(PREPROCESSED_DIR / "cycle_manifest_validated.csv")
valid_cycle_ids = manifest[~manifest["excluded"]]["cycle_id"].tolist()
print(f"{len(valid_cycle_ids)} valid cycles to process")

FEATURE_COLS = ["temperature_C", "humidity_pct", "soil_moisture_pct", "moisture_trend", "light_lux", "hour_of_day"]

4 valid cycles to process


In [2]:
def load_valid_cycles(valid_ids, raw_dir):
    cycles = {}
    for f in sorted(raw_dir.glob("*.csv")):
        cycle_id = f.stem.replace("_DUMMY", "")
        if cycle_id in valid_ids:
            df = pd.read_csv(f, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
            cycles[cycle_id] = df
    return cycles

cycles = load_valid_cycles(valid_cycle_ids, RAW_DIR)
print(f"Loaded {len(cycles)} valid cycles")

Loaded 4 valid cycles


## Compute moisture trend

`trend_t = moisture_t - moisture_(t-3)` — captures depletion **rate**, not just current level.

The first 2–3 hours of each cycle don't have enough history for a valid 3-hour trend. We drop
those rows rather than fabricate a trend value — a cycle's early hours are a small fraction of
its total labeled rows, so this costs little.

In [3]:
TREND_WINDOW = 3

def add_moisture_trend(df, window=TREND_WINDOW):
    df = df.copy()
    df["moisture_trend"] = df["soil_moisture_pct"] - df["soil_moisture_pct"].shift(window)
    return df

for cycle_id in cycles:
    cycles[cycle_id] = add_moisture_trend(cycles[cycle_id])

example = list(cycles.keys())[0]
cycles[example][["timestamp", "soil_moisture_pct", "moisture_trend"]].head(6)

,timestamp,soil_moisture_pct,moisture_trend
0,2026-07-01 09:00:00,87.11,NaN
1,2026-07-01 10:00:00,84.39,NaN
2,2026-07-01 11:00:00,82.01,NaN
3,2026-07-01 12:00:00,79.65,-7.46
4,2026-07-01 13:00:00,77.56,-6.83
5,2026-07-01 14:00:00,75.50,-6.51


## Extract hour-of-day

Kept as a raw numeric value (0-23), not pre-bucketed into morning/afternoon/etc. — lets the decision tree learn its own thresholds from real data rather than imposed boundaries.

In [4]:
for cycle_id in cycles:
    cycles[cycle_id]["hour_of_day"] = cycles[cycle_id]["timestamp"].dt.hour

cycles[example][["timestamp", "hour_of_day"]].head(6)

,timestamp,hour_of_day
0,2026-07-01 09:00:00,9
1,2026-07-01 10:00:00,10
2,2026-07-01 11:00:00,11
3,2026-07-01 12:00:00,12
4,2026-07-01 13:00:00,13
5,2026-07-01 14:00:00,14


## Drop rows without a valid trend (start of each cycle)

In [5]:
rows_before = sum(len(df) for df in cycles.values())

for cycle_id in cycles:
    cycles[cycle_id] = cycles[cycle_id].dropna(subset=["moisture_trend"]).reset_index(drop=True)

rows_after = sum(len(df) for df in cycles.values())
print(f"Rows before dropping trend-less rows: {rows_before}")
print(f"Rows after: {rows_after}  (dropped {rows_before - rows_after})")

Rows before dropping trend-less rows: 94
Rows after: 82  (dropped 12)


## Assemble the full feature table and save

In [6]:
feature_frames = []
for cycle_id, df in cycles.items():
    keep = df[["timestamp", "cycle_id", "condition"] + FEATURE_COLS].copy()
    feature_frames.append(keep)

features_df = pd.concat(feature_frames, ignore_index=True)

assert features_df[FEATURE_COLS].isnull().sum().sum() == 0, "Unexpected missing values remain!"

print(features_df.shape)
features_df.head()

(82, 9)


,timestamp,cycle_id,condition,temperature_C,humidity_pct,soil_moisture_pct,moisture_trend,light_lux,hour_of_day
0,2026-07-01 12:00:00,indoor_cycle01,indoor,27.65,47.72,79.65,-7.46,155.5,12
1,2026-07-01 13:00:00,indoor_cycle01,indoor,26.84,53.37,77.56,-6.83,170.5,13
2,2026-07-01 14:00:00,indoor_cycle01,indoor,28.18,52.27,75.50,-6.51,186.0,14
3,2026-07-01 15:00:00,indoor_cycle01,indoor,26.73,53.51,72.95,-6.70,155.3,15
4,2026-07-01 16:00:00,indoor_cycle01,indoor,26.39,53.02,70.03,-7.53,158.0,16


In [7]:
features_df.to_csv(PREPROCESSED_DIR / "features_only.csv", index=False)
print("Saved features_only.csv —", len(features_df), "rows across", features_df['cycle_id'].nunique(), "cycles")

Saved features_only.csv — 82 rows across 4 cycles


**Next step:** `04_label_construction.ipynb` — attach the urgency-bucket label to every row using each cycle's actual time-to-threshold.